In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/MyDrive/PhD/fedlbs_IWCMC/forecast-fedlbs')

In [ ]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMNet, GRUNet
from algorithm import fedlbs

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')

else:
    device = torch.device("cpu")

running on gpu


# Parameters

In [ ]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 256
n_layers = 2
lr = 0.001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training
num_clients_per_round = 15 #number of clients to participate in training

# Data Preperation

In [ ]:
dataframes = fetch_dataset()

Floor7
Floor3
Floor5
Floor4
Floor6
Floor6
Floor3
Floor5
Floor4
Floor7
Floor21
Floor16
Floor20
Floor21
Floor10
Floor15
Floor20
Floor14
Floor9
Floor17
Floor13
Floor11
Floor18
Floor19
Floor22
Floor8
Floor22
Floor9
Floor8
Floor18
Floor19
Floor14
Floor12
Floor16
Floor17
Floor15
Floor11
Floor13
Floor12
Floor10


In [ ]:
n_clients = len(dataframes) #total number of clients to partition data into

In [ ]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', '

# Build the training set

In [ ]:
train_loader = []
test_loader = []
label_scalers = []
i= 0
for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features
    if i ==15:
      break
    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )

    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler()

    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))

    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])

    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)

    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to(device), torch.from_numpy(train_y).to(device))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to(device), torch.from_numpy(test_y).to(device))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch

    # release some memory
    del train_x, train_y, test_x, test_y, train_data, test_data
    i=i+1

input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(68823, 90, 23) (68823, 1)
(40806, 90, 23) (40806, 1)
(72774, 90, 23) (72774, 1)
(40806, 90, 23) (40806, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)
(40806, 90, 23) (40806, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)
(72774, 90, 23) (72774, 1)


# LSTM model

In [ ]:
lstm = LSTMNet(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

LSTMNet(
  (lstm): LSTM(23, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (relu): ReLU()
)
814337


## S =3

In [ ]:
%%time
S = 3
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [18  1 19  8 10 17  6 13  4  2  5 14  9  7 16]
round 0, starting client 1/15, id: 18
Epoch [1/1], Train Loss: 0.029275559770615134
round 0, starting client 2/15, id: 1
Epoch [1/1], Train Loss: 0.013903395473342425
round 0, starting client 3/15, id: 19
Epoch [1/1], Train Loss: 0.016880311142485286
round 0, starting client 4/15, id: 8
Epoch [1/1], Train Loss: 0.018136064562642166
round 0, starting client 5/15, id: 10
Epoch [1/1], Train Loss: 0.017149901887366786
round 0, starting client 6/15, id: 17
Epoch [1/1], Train Loss: 0.025081395197870032
round 0, starting client 7/15, id: 6
Epoch [1/1], Train Loss: 0.01398013197675547
round 0, starting client 8/15, id: 13
Epoch [1/1], Train Loss: 0.14240403680014424
round 0, starting client 9/15, id: 4
Epoch [1/1], Train Loss: 0.01644801782094874
round 0, starting client 10/15, id: 2
Epoch [1/1], Train Loss: 0.017803852695838673
round 0, starting client 11/15, id: 5
Epoch [1/1], Train Loss: 0.017502605387042947


In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

/usr/local/lib/python3.10/dist-packages/numpy/lib/npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


## S = 5

In [ ]:
%%time
S = 5
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [11  1 18 17  2 12 19 16 10  0  3  4 15  8 13]
round 0, starting client 1/15, id: 11
Epoch [1/1], Train Loss: 0.028791739452691353
round 0, starting client 2/15, id: 1
Epoch [1/1], Train Loss: 0.01389788969106017
round 0, starting client 3/15, id: 18
Epoch [1/1], Train Loss: 0.016912160177913606
round 0, starting client 4/15, id: 17
Epoch [1/1], Train Loss: 0.01790211904940928
round 0, starting client 5/15, id: 2
Epoch [1/1], Train Loss: 0.017169443793805705
round 0, starting client 6/15, id: 12
Epoch [1/1], Train Loss: 0.025167393475590695
round 0, starting client 7/15, id: 19
Epoch [1/1], Train Loss: 0.013984078836594045
round 0, starting client 8/15, id: 16
Epoch [1/1], Train Loss: 0.1422609634718092
round 0, starting client 9/15, id: 10
Epoch [1/1], Train Loss: 0.01647042742946984
round 0, starting client 10/15, id: 0
Epoch [1/1], Train Loss: 0.017793604966885014
round 0, starting client 11/15, id: 3
Epoch [1/1], Train Loss: 0.01752711107422199
r

In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

/usr/local/lib/python3.10/dist-packages/numpy/lib/npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


## S= 7

In [ ]:
%%time
S = 7
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [15 13 12  5 11  2  8  6  3 17  4 10 16 18  9]
round 0, starting client 1/15, id: 15
Epoch [1/1], Train Loss: 0.028835400734783797
round 0, starting client 2/15, id: 13
Epoch [1/1], Train Loss: 0.013912014003164854
round 0, starting client 3/15, id: 12
Epoch [1/1], Train Loss: 0.016846042840176677
round 0, starting client 4/15, id: 5
Epoch [1/1], Train Loss: 0.017651282401887125
round 0, starting client 5/15, id: 11
Epoch [1/1], Train Loss: 0.01718010466088945
round 0, starting client 6/15, id: 2
Epoch [1/1], Train Loss: 0.026887542493040522
round 0, starting client 7/15, id: 8
Epoch [1/1], Train Loss: 0.014003776320480804
round 0, starting client 8/15, id: 6
Epoch [1/1], Train Loss: 0.14188334540719347
round 0, starting client 9/15, id: 3
Epoch [1/1], Train Loss: 0.016501073704733114
round 0, starting client 10/15, id: 17
Epoch [1/1], Train Loss: 0.017839278277408863
round 0, starting client 11/15, id: 4
Epoch [1/1], Train Loss: 0.017448433940964084

In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

/usr/local/lib/python3.10/dist-packages/numpy/lib/npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


## S=10

In [ ]:
%%time
S = 10
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [18  0 13  2  3 17  7 12 14 16 11 10  9 15  1]
round 0, starting client 1/15, id: 18
Epoch [1/1], Train Loss: 0.029005645587276963
round 0, starting client 2/15, id: 0
Epoch [1/1], Train Loss: 0.013914227310198812
round 0, starting client 3/15, id: 13
Epoch [1/1], Train Loss: 0.016898935592015746
round 0, starting client 4/15, id: 2
Epoch [1/1], Train Loss: 0.01807395959646083
round 0, starting client 5/15, id: 3
Epoch [1/1], Train Loss: 0.01711384730659566
round 0, starting client 6/15, id: 17
Epoch [1/1], Train Loss: 0.024124066681495956
round 0, starting client 7/15, id: 7
Epoch [1/1], Train Loss: 0.014005785680118222
round 0, starting client 8/15, id: 12
Epoch [1/1], Train Loss: 0.1420713433518916
round 0, starting client 9/15, id: 14
Epoch [1/1], Train Loss: 0.016499478681777997
round 0, starting client 10/15, id: 16
Epoch [1/1], Train Loss: 0.017781547823688012
round 0, starting client 11/15, id: 11
Epoch [1/1], Train Loss: 0.01746612019321349


In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}_C{num_clients_per_round}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}_C{num_clients_per_round}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}_C{num_clients_per_round}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}_C{num_clients_per_round}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}_C{num_clients_per_round}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}_C{num_clients_per_round}.npy', rmse_lbs)

/usr/local/lib/python3.10/dist-packages/numpy/lib/npyio.py:521: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)
